In [9]:
import andes
import numpy as np
import pandas as pd
import os
import random

import matplotlib
matplotlib.use('TkAgg')
import matplotlib.pyplot as plt

import scipy.io as sio

#Define which model to use
MODEL = 'ANDES_Dutch_2035_mpc_ID.xlsx'

In [ ]:
# Prepare andes for identification
ss = andes.prepare(models=['REGF1W_MPC_ID'])

### Run identification experiments

In [10]:
def gen_id_data(target_idx, size, experiment_type, extra_inv):
    dt = 0.01          # Simulation time step
    t_event = 10       # Time of the first event
    t_fin = 30         # End time of the simulation

    ss = andes.main.load(MODEL, setup=False, config_path="config_file.rc")

    n_units = 8
    inverter_ids = [43, 44, 52, 53, 54, 55, 56, 57]
    bus_map = {43: 28, 44: 28, 52: 28, 53: 11, 54: 28, 55: 22, 56: 22, 57: 21}
    load_map = {21: 1, 8: 2, 16: 3, 5: 4, 20: 5, 9: 6, 17: 7, 0: 8}

    # ------------------------------------------------------------------
    # Run the selected experiment
    # ------------------------------------------------------------------
    if experiment_type in {"u_step", "d", "du"}:
        for i in range(n_units):
            ss.REGF1W_MPC_ID.PRBS.v[i] = 0

        ss.setup()
        ss.PFlow.run()

        ss.TDS.config.tstep = dt
        ss.TDS.config.tf = t_event
        ss.TDS.run()

        if experiment_type == "u_step":
            ss.REGF1W_MPC_ID.P_ID.v[target_idx] += size
            ss.TDS.config.tf = t_fin
            ss.TDS.run()

        elif experiment_type == "d":
            ss.PQ.Ppf.v[target_idx] += size
            ss.TDS.config.tf = t_fin
            ss.TDS.run()

        elif experiment_type == "du":
            ss.PQ.Ppf.v[target_idx] += size
            ss.TDS.config.tf = 20
            ss.TDS.run()

            ss.REGF1W_MPC_ID.P_ID.v[extra_inv] += 0.03
            ss.TDS.config.tf = t_fin
            ss.TDS.run()

    elif experiment_type == "u_prbs":
        for i in range(n_units):
            ss.REGF1W_MPC_ID.PRBS.v[i] = 1
            ss.REGF1W_MPC_ID.seed.v[i] = target_idx

        ss.setup()
        ss.PFlow.run()

        ss.TDS.config.tstep = dt
        ss.TDS.config.tf = t_fin
        ss.TDS.run()

    elif experiment_type == "du_prbs":
        for i in range(n_units):
            ss.REGF1W_MPC_ID.PRBS.v[i] = 1
            ss.REGF1W_MPC_ID.seed.v[i] = target_idx

        # Random disturbance events are applied repeatedly during the run.
        event_times = np.linspace(t_event, 10 * int(t_fin * 0.1), int(t_fin * 0.1))

        ss.setup()
        ss.PFlow.run()

        ss.TDS.config.tstep = dt
        random.seed(target_idx)

        random_events = {}
        Pload_0 = [0] * n_units

        for i, event_t in enumerate(event_times):
            ss.TDS.config.tf = event_t
            ss.TDS.run()

            # Record the initial load values at the first event time.
            if i == 0:
                Pload_0[0] = ss.PQ.Ppf.v[21]
                Pload_0[1] = ss.PQ.Ppf.v[8]
                Pload_0[2] = ss.PQ.Ppf.v[16]
                Pload_0[3] = ss.PQ.Ppf.v[5]
                Pload_0[4] = ss.PQ.Ppf.v[20]
                Pload_0[5] = ss.PQ.Ppf.v[9]
                Pload_0[6] = ss.PQ.Ppf.v[17]
                Pload_0[7] = ss.PQ.Ppf.v[0]

            sizes = [random.choice([0.2, 0.0]) for _ in range(n_units)]
            random_events[event_t] = sizes

            ss.PQ.Ppf.v[21] = Pload_0[0] + sizes[0]
            ss.PQ.Ppf.v[8]  = Pload_0[1] + sizes[1]
            ss.PQ.Ppf.v[16] = Pload_0[2] + sizes[2]
            ss.PQ.Ppf.v[5]  = Pload_0[3] + sizes[3]
            ss.PQ.Ppf.v[20] = Pload_0[4] + sizes[4]
            ss.PQ.Ppf.v[9]  = Pload_0[5] + sizes[5]
            ss.PQ.Ppf.v[17] = Pload_0[6] + sizes[6]
            ss.PQ.Ppf.v[0]  = Pload_0[7] + sizes[7]

        ss.TDS.config.tf = t_fin
        ss.TDS.run()

    else:
        raise ValueError(
            f"Unknown experiment type '{experiment_type}'. "
            "Use one of: u_step, u_prbs, d, du, du_prbs."
        )

    # ------------------------------------------------------------------
    # Extract and organize the simulation output
    # ------------------------------------------------------------------
    df = ss.dae.ts.df
    time = df.index

    gen_df = pd.read_excel(MODEL, sheet_name="GENROU")
    f_df = df.filter(regex=r"^f BusROCOF")
    bus_numbers = f_df.columns.str.extract(r"(\d+)").astype(int)[0]

    # Center-of-inertia frequency calculation.
    gen_df["weight"] = gen_df["Sn"] * (gen_df["M"] / 2)
    bus_weights = gen_df.groupby("bus")["weight"].sum()
    weights = bus_numbers.map(bus_weights).fillna(0.0).to_numpy()
    f_COI = (f_df.to_numpy() @ weights) / weights.sum()

    t = np.around(np.asarray(time), 3)
    mask = ~pd.Series(t).duplicated()

    data = {}
    data["h"] = dt
    data["t"] = t[mask].reshape(-1, 1)

    zero_signal = np.zeros_like(time)

    if experiment_type in {"d", "du"}:
        disturbance_signal = np.zeros_like(time)
        which_d = load_map[target_idx]
        disturbance_signal[time >= t_event] = size

        data[f"d_{which_d}"] = disturbance_signal[mask].reshape(-1, 1)

        for i in range(1, 9):
            if i != which_d:
                data[f"d_{i}"] = zero_signal[mask].reshape(-1, 1)

    elif experiment_type == "du_prbs":
        d_sigs = [np.zeros_like(time) for _ in range(n_units)]

        for event_t in event_times:
            sizes = random_events[event_t]
            for i in range(n_units):
                d_sigs[i][time >= event_t] = sizes[i]

        for i in range(n_units):
            data[f"d_{i + 1}"] = d_sigs[i][mask].reshape(-1, 1)

    else:
        for i in range(n_units):
            data[f"d_{i + 1}"] = zero_signal[mask].reshape(-1, 1)

    u_df = df.filter(regex=r"^P_id REGF1W MPC ID")
    P_df = df.filter(regex=r"^Pe REGF1W MPC ID")

    data["f_COI"] = (50 * f_COI - 50)[mask].reshape(-1, 1)

    for inv_id in inverter_ids:
        data[f"Pmpc_{inv_id}"] = (
            u_df[f"P_id REGF1W MPC ID {inv_id}"].to_numpy()[mask].reshape(-1, 1)
        )
        data[f"f_{inv_id}"] = (
            50 * df[f"f BusROCOF {bus_map[inv_id]}"] - 50
        ).to_numpy()[mask].reshape(-1, 1)
        data[f"Pe_{inv_id}"] = (
            P_df[f"Pe REGF1W MPC ID {inv_id}"] - P_df[f"Pe REGF1W MPC ID {inv_id}"].iloc[0]
        ).to_numpy()[mask].reshape(-1, 1)

    for i, col in enumerate(f_df.columns):
        data[f"f_bus_{i}"] = (50 * df[col] - 50).to_numpy()[mask].reshape(-1, 1)

    # ------------------------------------------------------------------
    # Save the generated dataset
    # ------------------------------------------------------------------
    if experiment_type == "u_step":
        inv = inverter_ids[target_idx]
        sio.savemat(f"id_data/IBR={inv}_u={size}.mat", data)

    elif experiment_type == "d":
        sio.savemat(f"id_data/loc={target_idx}_d={size}.mat", data)

    elif experiment_type == "u_prbs":
        sio.savemat(f"id_data/u=PRBS_{int(target_idx)}.mat", data)

    elif experiment_type == "du_prbs":
        sio.savemat(f"id_data/du=PRBS_{int(target_idx)}.mat", data)

    elif experiment_type == "du":
        sio.savemat(f"id_data/IBR={inverter_ids[extra_inv]}_loc={target_idx}.mat", data)

    return

#### Input step experiments

In [11]:
# Input step experiments
invs = [0, 1, 2, 3, 4, 5, 6, 7]

experiments = []

for inv in invs:
    experiments = experiments + [[inv,-0.02,'u_step',0]]
    experiments = experiments + [[inv,-0.03,'u_step',0]]
    experiments = experiments + [[inv,-0.04,'u_step',0]]


for exp in experiments:
    gen_id_data(exp[0], exp[1], exp[2], exp[3])
    print(f'Finished with exp = {exp}')

  0%|                                                    | 0/100 [00:00<?, ?%/s]

  0%|                                                    | 0/100 [00:00<?, ?%/s]

Finished with exp = [0, -0.02, 'u_step', 0]


#### Disturbance step experiments

In [ ]:
# Disturbance step experiments
locs = [21,8,16,5,20,9,17,0]

for loc in locs:
    experiments = experiments + [[loc,0.2,'d',0]]
    experiments = experiments + [[loc,0.3,'d',0]]

for exp in experiments:
    gen_id_data(exp[0], exp[1], exp[2], exp[3])
    print(f'Finished with exp = {exp}')

#### PRBS experiments

In [ ]:
# PRBS experiments
u_seeds = [1,2,3,4,5,6,7,8,9,10,11,12,13,14,15]
d_seeds = [4,5,6,7,8,9,10]
experiments = []

# PRBS experiments on just inputs
for seed in u_seeds:
     experiments = experiments + [[seed,0,'u_prbs',0]]
# PRBS experiments on both input and disturbance channels
for seed in d_seeds:
     experiments = experiments + [[seed,0,'du_prbs',0]]


for exp in experiments:
    gen_id_data(exp[0], exp[1], exp[2], exp[3])
    print(f'Finished with exp = {exp}')